# XGBoost 再学習ノートブック v2

| セル | 内容 |
|------|------|
| セル1 | Google Drive マウント |
| セル2 | src/ 強制アップデート（GitHub最新） |
| セル3 | speed_indexキャッシュ再構築 → 学習データ生成 |
| セル4 | XGB再学習 → キャリブレーション |
| セル5 | 統合テスト（cal_prob合計・重要度確認） |

In [ ]:
# ── セル1: セットアップ ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE_DIR = '/content/drive/MyDrive/keiba_ai'
sys.path.insert(0, BASE_DIR)
print(f'✅ BASE_DIR: {BASE_DIR}')

In [ ]:
# ── セル2: src/ 強制アップデート（GitHub最新コードを取得）────────
import urllib.request, time as _time
BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
_cb = int(_time.time())
files = [
    'src/tools/__init__.py', 'src/tools/tune_weights.py',
    'src/tools/calibrate.py', 'src/tools/analyze_divergence.py',
    'src/tools/rescrape_history.py', 'src/tools/build_training_data.py',
    'src/tools/train_xgb.py', 'src/tools/calibrate_xgb.py',
    'src/features/engine.py', 'src/features/speed_index.py',
    'src/utils/config.py', 'src/utils/db.py', 'src/utils/model_registry.py',
    'src/scraper/parser.py', 'src/scraper/jra_scraper.py',
    'src/models/__init__.py', 'src/models/calibration.py',
    'src/models/calibration_xgb.py', 'src/models/predict.py',
    'src/betting/__init__.py', 'src/betting/make_bets.py',
    'src/betting/ev_filter.py', 'src/betting/app_json.py',
    'src/betting/shadow.py',
]
for rel in files:
    dest = f'{BASE_DIR}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    urllib.request.urlretrieve(f'{BASE_URL}/{rel}?nocache={_cb}', dest)
    print(f'OK {rel}')
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

# 主要特徴量の存在確認
with open(f'{BASE_DIR}/src/features/engine.py') as _f:
    _eng_src = _f.read()
for _kw in ['f_member_level_avg', 'f_speed_fig_last', 'f_speed_fig_avg', 'f_finish_time_avg', '±200']:
    _ok = _kw in _eng_src
    print(f'  {"✅" if _ok else "❌"} engine.py に {_kw} {"あり" if _ok else "なし! ← 要確認"}')
print('done')

In [ ]:
# ── セル3: speed_index キャッシュ再構築 → 学習データ生成 ────────
# speed_index_cache.pkl を最新 history.db から再構築してから CSV 生成
from src.features.speed_index import rebuild_speed_index_cache
rebuild_speed_index_cache(BASE_DIR)

for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.tools.build_training_data import build_training_data
df_all = build_training_data(BASE_DIR)

# 特徴量充足率確認
print('\n── 特徴量充足率確認 ──')
CHECK_COLS = [
    'f_member_level_avg', 'f_finish_time_avg', 'f_time_diff_avg',
    'f_speed_fig_last', 'f_speed_fig_avg', 'f_speed_fig_max',
]
for c in CHECK_COLS:
    if c in df_all.columns:
        pct = 100 * df_all[c].notna().sum() / len(df_all)
        print(f'  ✅ {c}: {pct:.1f}%')
    else:
        print(f'  ❌ {c}: 列なし')
print(f'\n総列数: {len(df_all.columns)}  総行数: {len(df_all)}')

In [ ]:
# ── セル4: XGB再学習 → キャリブレーション ───────────────────────
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.tools.train_xgb import train_xgb
from src.tools.calibrate_xgb import run_xgb_calibration

# ① 再学習
metrics = train_xgb(BASE_DIR,
    train_end='2026-05-31',
    val_start='2026-06-01',
    val_end='2026-06-15')
print(metrics)

# ② AUC 0.75以上ならキャリブレーション
if metrics.get('auc', 0) >= 0.75:
    run_xgb_calibration(BASE_DIR)
    print('✅ キャリブレーション完了')
else:
    print(f'⚠ AUC={metrics.get("auc", 0):.4f} < 0.75 → キャリブレーションスキップ')

In [ ]:
# ── セル5: 統合テスト ──────────────────────────────────────────
import pickle, json, numpy as np, pandas as pd

with open(f'{BASE_DIR}/data/xgb_fukusho_model.pkl', 'rb') as f: model = pickle.load(f)
with open(f'{BASE_DIR}/data/xgb_feature_cols.json')        as f: cols  = json.load(f)['feature_cols']
with open(f'{BASE_DIR}/data/xgb_calibrator.pkl', 'rb')     as f: calib = pickle.load(f)

print(f'✅ モデル特徴量数: {len(cols)}')
print(f'✅ キャリブレーター: {type(calib).__name__}')

# ダミー16頭で cal_prob 合計確認
dummy = pd.DataFrame([{c: np.random.normal(5.0, 1.5) for c in cols} for _ in range(16)])
raw_probs = model.predict_proba(dummy)[:, 1]
cal_probs = np.array(calib.transform(raw_probs))

print(f'\n── ダミー16頭テスト ──')
print(f'  raw_prob range: {raw_probs.min():.3f} ~ {raw_probs.max():.3f}')
print(f'  cal_prob 合計: {cal_probs.sum():.3f}  (2.8〜3.2 が正常)')
print(f'  合計チェック  {"✅" if 2.8 <= cal_probs.sum() <= 3.2 else "⚠ 異常値"}')
print(f'  分散チェック  {"✅" if raw_probs.max() - raw_probs.min() > 0.3 else "⚠ 分散小さい"}')

# 特徴量重要度トップ20
_speed_fig = {'f_speed_fig_last', 'f_speed_fig_avg', 'f_speed_fig_max'}
_member    = {'f_member_level_avg', 'f_member_level_max', 'f_member_level_last'}
_time_feat = {'f_finish_time_avg', 'f_time_diff_avg'}
imps = sorted(zip(cols, model.feature_importances_), key=lambda x: x[1], reverse=True)[:20]
print('\n── 特徴量重要度 Top 20 ──')
for name, imp in imps:
    if name in _speed_fig: mark = '★スピード指数'
    elif name in _member:  mark = '★メンバーLv'
    elif name in _time_feat: mark = '★タイム(±200m)'
    else: mark = ''
    print(f'  {name:<42} {imp*100:.2f}% {mark}')